In [2]:
# Import the libraries
import numpy as np
import plotly.express as px
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression

In [3]:
# Define exact filenames
co2_file = 'owid-co2-data.csv'
# Using the exact name of the file you provided
happiness_file = 'WHR25_Data_Figure_2.1v3.xlsx'

#  Load CO2 Data (Standard UTF-8)
df_co2 = pd.read_csv(co2_file)

#  Load Happiness Data 
df_hap = pd.read_excel(happiness_file)

# 4. Standardize Columns
df_hap = df_hap.rename(columns={
    'Country name': 'country',
    'Year': 'year',
    'Life evaluation (3-year average)': 'life_satisfaction'
})

# Keep essential CO2 columns
co2_subset = df_co2[['country', 'year', 'iso_code', 'co2_per_capita', 'population', 'gdp']]

# Using an inner join to ensure we have matches for both datasets
df_merged = pd.merge(df_hap, co2_subset, on=['country', 'year'], how='inner')

# Save the result
df_merged.to_csv('merged_happiness_co2.csv', index=False)

print("Successfully merged!")
print(f"Final dataset contains {len(df_merged)} rows.")
print(df_merged[['country', 'year', 'life_satisfaction', 'co2_per_capita']].head())

Successfully merged!
Final dataset contains 1810 rows.
       country  year  life_satisfaction  co2_per_capita
0  Afghanistan  2024              1.364           0.254
1  Afghanistan  2023              1.721           0.254
2  Afghanistan  2022              1.859           0.251
3  Afghanistan  2021              2.404           0.247
4  Afghanistan  2020              2.523           0.285


##### Figure 4,  Efficiency of CO2 emissions, finding which countries get most happiness and least happiness per unit of carbon emissions scatterplot

Countries with labels are particular outliers

In [ ]:
#Load the merged dataset -
df = pd.read_csv('merged_happiness_co2.csv')
df_2022 = df[df['year'] == 2022].copy()

# Calculate Efficiency and Rankings
df_2022 = df_2022[df_2022['co2_per_capita'] > 0].dropna(subset=['life_satisfaction', 'co2_per_capita'])
df_2022['efficiency'] = df_2022['life_satisfaction'] / df_2022['co2_per_capita']

# Regression for Residuals - this is how we define efficiency rankings
X = np.log10(df_2022[['co2_per_capita']])
y = df_2022['efficiency']
reg = LinearRegression().fit(X, y)
df_2022['residuals'] = y - reg.predict(X)
df_2022['residual_pct'] = df_2022['residuals'].rank(pct=True) * 100

# Define Status Categories
q_low, q_high = df_2022['residuals'].quantile([0.33, 0.66])
def get_status(res):
    if res > q_high: return "High Efficiency"
    if res < q_low: return "Low Efficiency"
    return "Average"
df_2022['Efficiency Status'] = df_2022['residuals'].apply(get_status)

#  binding_range for the slider
slider = alt.binding_range(min=0, max=100, step=1, name='Min Efficiency Percentile: ')

#  setting the starting point
eff_selector = alt.selection_point(
    name='EffFilter', 
    fields=['residual_pct'], 
    bind=slider,
    value=[{'residual_pct': 0}]
)

# Define the base chart - adding parameters later
base = alt.Chart(df_2022).transform_filter(
    alt.datum.residual_pct >= eff_selector.residual_pct
)

# Scatter Layer
scatter = base.mark_point(filled=True, size=100, opacity=0.7).encode(
    x=alt.X('co2_per_capita:Q', scale=alt.Scale(type='log'), title='CO2 per Capita (Tonnes, Log Scale)'),
    y=alt.Y('efficiency:Q', title='Efficiency (Life Satisfaction / CO2)'),
    color=alt.Color('Efficiency Status:N', 
                    scale=alt.Scale(domain=['High Efficiency', 'Global Average', 'Low Efficiency'], 
                                    range=['#20b2aa', '#778899', '#ff6347'])),
    tooltip=['country', 'life_satisfaction', 'co2_per_capita', 'efficiency']
)

# Reference Line Layer
avg_eff = df_2022['efficiency'].mean()
rule = alt.Chart(pd.DataFrame({'y': [avg_eff]})).mark_rule(
    strokeDash=[5,5], color='black', opacity=0.5
).encode(y='y:Q')

# Annotation for average line
avg_label = alt.Chart().mark_text(
    align='right',
    baseline='bottom',
    dx=-5,
    dy=-5,
    fontSize=12,
    fontWeight='bold',
    color='black',
    text='Global Average Efficiency'
).encode(
    x=alt.value(600),     # Hard-coded to the right edge (600px)
    y=alt.datum(avg_eff)  # Hard-coded to the mean efficiency value
)

# Adding labels for outliers
outliers = ['Malawi', 'Costa Rica', 'Vietnam', 'Norway', 'United States', 'Qatar']
labels = base.mark_text(align='left', dx=7, dy=-7, fontWeight='bold').encode(
    x='co2_per_capita:Q',
    y='efficiency:Q',
    text='country:N'
).transform_filter(
    alt.FieldOneOfPredicate(field='country', oneOf=outliers)
)

#   add_params() on the final layered chart
final_chart = (scatter + rule + labels + avg_label).properties(   
    width=600,
    height=400,
    title='Visual #4: Efficiency of CO2 on Life Satisfaction (2022)'
).add_params(
    eff_selector
).interactive()

final_chart.display()

alt.LayerChart(...)

##### Visual #5: 10-Year Decoupling Trends

In [5]:
#  Load data
df = pd.read_csv('merged_happiness_co2.csv')

# Add Continent Mapping
def get_continent(iso):
    if pd.isna(iso): return 'Other'
    mapping = {
        'US': 'N. America', 'CA': 'N. America', 'MX': 'N. America', 
        'GB': 'Europe', 'FR': 'Europe', 'DE': 'Europe', 'NO': 'Europe', 'IT': 'Europe',
        'CH': 'Asia', 'IN': 'Asia', 'JP': 'Asia', 'VN': 'Asia', 'KR': 'Asia', 'CN': 'Asia',
        'BR': 'S. America', 'AR': 'S. America', 'CL': 'S. America',
        'ZA': 'Africa', 'EG': 'Africa', 'ET': 'Africa', 'KE': 'Africa',
        'AU': 'Oceania', 'NZ': 'Oceania'
    }
    return mapping.get(iso[:2], 'Global/Other')

if 'continent' not in df.columns:
    df['continent'] = df['iso_code'].apply(get_continent)

#  GLOBAL NORMALIZATION 
df_decade = df[df['year'].between(2014, 2024)].copy()

# Scale relative to the whole world's min/max for the decade
co2_min, co2_max = df_decade['co2_per_capita'].min(), df_decade['co2_per_capita'].max()
ls_min, ls_max = df_decade['life_satisfaction'].min(), df_decade['life_satisfaction'].max()

df_decade['norm_co2'] = 100 * (df_decade['co2_per_capita'] - co2_min) / (co2_max - co2_min)
df_decade['norm_ls'] = 100 * (df_decade['life_satisfaction'] - ls_min) / (ls_max - ls_min)

# Decoupling Logic (Green if Happy Up & CO2 Down, Red if both Up)
def calculate_status(group):
    v14 = group[group['year'] == 2014]
    v24 = group[group['year'] == 2024]
    if not v14.empty and not v24.empty:
        happy_up = v24['life_satisfaction'].values[0] > v14['life_satisfaction'].values[0]
        co2_down = v24['co2_per_capita'].values[0] < v14['co2_per_capita'].values[0]
        co2_up = v24['co2_per_capita'].values[0] > v14['co2_per_capita'].values[0]
        
        if happy_up and co2_down:
            return 'Success' # Green
        elif happy_up and co2_up:
            return 'Coupled' # Red
    return 'Other'

# Apply status
status_map = df_decade.groupby('country').apply(calculate_status, include_groups=False).to_dict()
df_decade['status'] = df_decade['country'].map(status_map)

# Melt for Visualization
df_melted = df_decade.melt(
    id_vars=['country', 'year', 'status', 'continent', 'life_satisfaction', 'co2_per_capita'],
    value_vars=['norm_co2', 'norm_ls'],
    var_name='Metric', value_name='Score'
)
df_melted['Metric_Label'] = df_melted['Metric'].map({'norm_co2': 'Carbon (CO2)', 'norm_ls': 'Life Satisfaction'})


# Interaction: Region Filter (Dropdown)
continents = sorted([c for c in df_melted['continent'].unique() if c])
continent_select = alt.binding_select(options=continents + [None], labels=continents + ['All Regions'], name='Show Region: ')
continent_param = alt.selection_point(fields=['continent'], bind=continent_select, name='continent_param')

# Interaction: Click to Expand
expand_param = alt.selection_point(fields=['country'], on='click', clear='dblclick', toggle=False, name='expand_param')

# Base chart with the Region Filter applied UPSTREAM
base = alt.Chart(df_melted).transform_filter(continent_param)

# Background Layer
background = base.mark_rect(opacity=0.15).encode(
    color=alt.Color('status:N', 
                    scale=alt.Scale(domain=['Success', 'Coupled', 'Other'], 
                                    range=['#2ecc71', '#e74c3c', '#d5dbdb']),
                    legend=alt.Legend(title="Trend"))
)

# Line Layer (High Contrast Colors)
lines = base.mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('Score:Q', title='Global Scale (0-100)', scale=alt.Scale(domain=[-5, 105])),
    color=alt.Color('Metric_Label:N', 
                    scale=alt.Scale(range=['#2c3e50', '#f39c12']), # Dark Blue vs Bright Orange
                    legend=alt.Legend(title="Indicator")),
    tooltip=[
        alt.Tooltip('country:N'),
        alt.Tooltip('year:O'),
        alt.Tooltip('life_satisfaction:Q', format='.2f', title='Happiness (Raw)'),
        alt.Tooltip('co2_per_capita:Q', format='.2f', title='CO2 (Raw)'),
        alt.Tooltip('Score:Q', format='.1f', title='Normalized Score')
    ]
).transform_filter(
    # Expand path only for clicked country
    "(datum.year == 2014 || datum.year == 2024) || (expand_param.country == datum.country)"
).add_params(expand_param)

# Final Layout
chart = (background + lines).facet(
    facet=alt.Facet('country:N', title=None, header=alt.Header(labelFontSize=12)),
    columns=4
).resolve_scale(
    x='independent'
).add_params(
    continent_param
).properties(
    title=alt.TitleParams(
        text='Visual #5: 10-Year Decoupling Trends',
        subtitle='Green: Happiness ↑ & Carbon ↓ | Red: Happiness ↑ & Carbon ↑ (Click country to see full 10-year path)'
    )
)

chart.display()

alt.FacetChart(...)

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load the data
df = pd.read_csv('merged_happiness_co2.csv')

# Continent Mapping
continent_map = {
    'AFG': 'Asia', 'ALB': 'Europe', 'DZA': 'Africa', 'AGO': 'Africa', 'ARG': 'S. America',
    'ARM': 'Asia', 'AUS': 'Oceania', 'AUT': 'Europe', 'AZE': 'Asia', 'BHR': 'Asia',
    'BGD': 'Asia', 'BLR': 'Europe', 'BEL': 'Europe', 'BLZ': 'S. America', 'BEN': 'Africa',
    'BTN': 'Asia', 'BOL': 'S. America', 'BIH': 'Europe', 'BWA': 'Africa', 'BRA': 'S. America',
    'BGR': 'Europe', 'BFA': 'Africa', 'BDI': 'Africa', 'KHM': 'Asia', 'CMR': 'Africa',
    'CAN': 'N. America', 'CAF': 'Africa', 'TCD': 'Africa', 'CHL': 'S. America', 'CHN': 'Asia',
    'COL': 'S. America', 'COM': 'Africa', 'COG': 'Africa', 'CRI': 'N. America', 'HRV': 'Europe',
    'CUB': 'N. America', 'CYP': 'Asia', 'CZE': 'Europe', 'DNK': 'Europe', 'DJI': 'Africa',
    'DOM': 'N. America', 'ECU': 'S. America', 'EGY': 'Africa', 'SLV': 'N. America', 'EST': 'Europe',
    'SWZ': 'Africa', 'ETH': 'Africa', 'FIN': 'Europe', 'FRA': 'Europe', 'GAB': 'Africa',
    'GMB': 'Africa', 'GEO': 'Asia', 'DEU': 'Europe', 'GHA': 'Africa', 'GRC': 'Europe',
    'GTM': 'N. America', 'GIN': 'Africa', 'GUY': 'S. America', 'HTI': 'N. America', 'HND': 'N. America',
    'HUN': 'Europe', 'ISL': 'Europe', 'IND': 'Asia', 'IDN': 'Asia', 'IRN': 'Asia',
    'IRQ': 'Asia', 'IRL': 'Europe', 'ISR': 'Asia', 'ITA': 'Europe', 'JAM': 'N. America',
    'JPN': 'Asia', 'JOR': 'Asia', 'KAZ': 'Asia', 'KEN': 'Africa', 'KOS': 'Europe',
    'KWT': 'Asia', 'KGZ': 'Asia', 'LVA': 'Europe', 'LBN': 'Asia', 'LSO': 'Africa',
    'LBR': 'Africa', 'LBY': 'Africa', 'LTU': 'Europe', 'LUX': 'Europe', 'MDG': 'Africa',
    'MWI': 'Africa', 'MYS': 'Asia', 'MDV': 'Asia', 'MLI': 'Africa', 'MLT': 'Europe',
    'MRT': 'Africa', 'MUS': 'Africa', 'MEX': 'N. America', 'MNG': 'Asia', 'MNE': 'Europe',
    'MAR': 'Africa', 'MOZ': 'Africa', 'MMR': 'Asia', 'NAM': 'Africa', 'NPL': 'Asia',
    'NLD': 'Europe', 'NZL': 'Oceania', 'NIC': 'N. America', 'NER': 'Africa', 'NGA': 'Africa',
    'MKD': 'Europe', 'NOR': 'Europe', 'OMN': 'Asia', 'PAK': 'Asia', 'PAN': 'N. America',
    'PRY': 'S. America', 'PER': 'S. America', 'PHL': 'Asia', 'POL': 'Europe', 'PRT': 'Europe',
    'QAT': 'Asia', 'ROU': 'Europe', 'RWA': 'Africa', 'SAU': 'Asia', 'SEN': 'Africa',
    'SRB': 'Europe', 'SLE': 'Africa', 'SGP': 'Asia', 'SVK': 'Europe', 'SVN': 'Europe',
    'SOM': 'Africa', 'ZAF': 'Africa', 'SSD': 'Africa', 'ESP': 'Europe', 'LKA': 'Asia',
    'SDN': 'Africa', 'SUR': 'S. America', 'SWE': 'Europe', 'CHE': 'Europe', 'SYR': 'Asia',
    'TJK': 'Asia', 'TZA': 'Africa', 'THA': 'Asia', 'TGO': 'Africa', 'TTO': 'S. America',
    'TUN': 'Africa', 'TKM': 'Asia', 'UGA': 'Africa', 'UKR': 'Europe', 'ARE': 'Asia',
    'GBR': 'Europe', 'USA': 'N. America', 'URY': 'S. America', 'UZB': 'Asia', 'VEN': 'S. America',
    'YEM': 'Asia', 'ZMB': 'Africa', 'ZWE': 'Africa'
}
df['continent'] = df['iso_code'].map(continent_map).fillna('Other')

# GLOBAL NORMALIZATION
df_decade = df[df['year'].between(2014, 2024)].copy()
g_co2_max = df_decade['co2_per_capita'].max()
g_ls_max = df_decade['life_satisfaction'].max()

df_decade['norm_co2'] = 100 * (df_decade['co2_per_capita'] / g_co2_max)
df_decade['norm_ls'] = 100 * (df_decade['life_satisfaction'] / g_ls_max)

# TREND STATUS
def get_status(group):
    g = group.sort_values('year')
    v14, v24 = g[g['year'] == 2014], g[g['year'] == 2024]
    if not v14.empty and not v24.empty:
        if v24['life_satisfaction'].iloc[0] > v14['life_satisfaction'].iloc[0] and \
           v24['co2_per_capita'].iloc[0] < v14['co2_per_capita'].iloc[0]: return 'Success'
        if v24['life_satisfaction'].iloc[0] > v14['life_satisfaction'].iloc[0]: return 'Coupled'
    return 'Other'

status_map = df_decade.groupby('country').apply(get_status).to_dict()
df_decade['status'] = df_decade['country'].map(status_map)

# INTERACTIVE WIDGET INTERFACE 
out = widgets.Output()
# Wrap output in a scrollable Box
scrollable_box = widgets.Box([out], layout={'height': '600px', 'overflow': 'auto', 'border': '1px solid #ccc'})

dropdown = widgets.Dropdown(options=sorted(df_decade['continent'].unique()), value='Europe', description='Region:')

def update_chart(change):
    with out:
        clear_output(wait=True)
        region_df = df_decade[df_decade['continent'] == change['new']].copy()
        melted = region_df.melt(id_vars=['country', 'year', 'status'], value_vars=['norm_co2', 'norm_ls'])
        
        # Define Selection
        expand_p = alt.selection_point(fields=['country'], on='click', name='expand_p', toggle=False)

        base = alt.Chart(melted).properties(width=160, height=120)

        # Background Layer (Catch clicks anywhere) // CHECK THIS
        bg = base.mark_rect(opacity=0.15).encode(
            color=alt.Color('status:N', scale=alt.Scale(domain=['Success', 'Coupled', 'Other'], 
                                                        range=['#2ecc71', '#e74c3c', '#d5dbdb']))
        ).add_params(expand_p)

        #  Invisible layer to make the entire facet area clickable
        clickable = base.mark_rect(opacity=0).encode(tooltip='country:N')

        # Line Layer
        lines = base.mark_line(point=True).encode(
            x=alt.X('year:O', title=None),
            y=alt.Y('value:Q', title='Global Scale', scale=alt.Scale(domain=[0, 100])),
            color=alt.Color('variable:N', scale=alt.Scale(range=['#2c3e50', '#f39c12']))
        ).transform_filter(
            (alt.datum.year == 2014) | (alt.datum.year == 2024) | expand_p
        )

        chart = (bg + clickable + lines).facet(facet='country:N', columns=4).resolve_scale(y='shared')
        chart.display()

dropdown.observe(update_chart, names='value')
display(dropdown, scrollable_box)
update_chart({'new': dropdown.value})

C:\Users\nicho\AppData\Local\Temp\ipykernel_30192\2334456045.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  status_map = df_decade.groupby('country').apply(get_status).to_dict()


Dropdown(description='Region:', index=2, options=('Africa', 'Asia', 'Europe', 'N. America', 'Oceania', 'Other'…

Box(children=(Output(),), layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_r…

## ^ STILL SO JANK - idk what happened to the clicking but filter works...